# TrimSQD Demo — N₂ / 6-31G

This notebook benchmarks **TrimSQD** against standard SQD on the same N₂ / 6-31G
active-space problem used in [qiskit-addon-sqd tutorial 01](https://qiskit.github.io/qiskit-addon-sqd/tutorials/01_chemistry_hamiltonian.html).

**TrimSQD** (PR #369 in qiskit-addon-sqd) improves convergence by screening a
larger candidate pool before diagonalization:

| Step | Standard SQD | TrimSQD |
|------|-------------|---------|
| Sampling | subsample each batch independently | draw one disjoint pool, partition |
| Screening | — | diagonalize each batch, trim to top-weight CI strings |
| Reporting | best-batch energy | merged-subspace diagonalization energy |
| Carryover | amplitude-threshold strings | trimmed merged-round strings |

Both runs use **identical QPU counts** — the job is submitted once and the
 checkpoint is shared.  The only difference is the classical
post-processing policy.

### System
- Molecule: N₂ at 1.0 Å (STO-3G equilibrium)
- Basis: 6-31G
- Active space: 8 orbitals, 10 electrons (2 frozen 1s core)
- QPU circuit: 16-qubit LUCJ ansatz on ibm_kingston
- Reference: CASCI = −109.04667800 Ha (qiskit-addon-sqd tutorial)


## Setup

In [1]:
import os
import yaml
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from pyscf import gto, scf, cc, mcscf, ao2mo

from quantum_fragment_methods.qpu import QRMIBackend
from quantum_fragment_methods.application.solvers.quantum_zoo.sqd import SQDSolver
from quantum_fragment_methods.application.solvers.quantum_zoo.trim_sqd import TrimSQDSolver

DEMO_DIR = Path('__file__').resolve().parent if '__file__' in dir() else Path().resolve()
print(f'Demo directory: {DEMO_DIR}')

Demo directory: /Users/thaddeuspellegrini/Code/QuantumAlgorithmEngineering/QDC_2026/quantum-fragment-methods/examples/notebook_demos/sqd_demos


## 1. Load Configuration

In [2]:
config_path = DEMO_DIR / 'config_N2_6-31G_trim_demo.yaml'

with open(config_path) as f:
    config = yaml.safe_load(f)

qpu_config  = config['qpu']
sqd_config  = config['sqd']
trim_config = sqd_config.get('trim', {})

print(f'Config     : {config_path}')
print(f'Backend    : {qpu_config["backend_name"]}')
print(f'Shots      : {qpu_config["sampler_options"]["default_shots"]}')
print(f'SQD iter   : {sqd_config["iterations"]}  batches: {sqd_config["n_batches"]}  samples: {sqd_config["samples_per_batch"]}')
print(f'Trim ratio : {trim_config.get("trim_ratio", 0.5)}')
print(f'Max strings: {trim_config.get("max_strings_per_trim", "None")}')
print(f'Policy     : {"SectorTrimPolicy" if trim_config.get("sector_trim") else "TrimPolicy"}')

Config     : /Users/thaddeuspellegrini/Code/QuantumAlgorithmEngineering/QDC_2026/quantum-fragment-methods/examples/notebook_demos/sqd_demos/config_N2_6-31G_trim_demo.yaml
Backend    : ibm_kingston
Shots      : 100000
SQD iter   : 5  batches: 5  samples: 300
Trim ratio : 0.5
Max strings: 250
Policy     : TrimPolicy


## 2. Define Molecular System — N₂ / 6-31G

Active space: freeze the two 1s core orbitals (matches qiskit-addon-sqd tutorial).


In [3]:
mol = gto.Mole()
mol.atom = '''
N 0.0 0.0 0.0
N 1.0 0.0 0.0
'''
mol.basis = '6-31G'
mol.build()

print(f'Orbitals : {mol.nao}')
print(f'Electrons: {mol.nelectron}')

Orbitals : 18
Electrons: 14


## 3. Hartree-Fock + Active-Space Hamiltonian

In [4]:
mf = scf.RHF(mol)
mf.kernel()
print(f'HF energy: {mf.e_tot:.8f}')

# Active space: freeze the two 1s core orbitals
n_frozen     = 2
active_space = range(n_frozen, mol.nao_nr())
norb         = len(active_space)
nelec_total  = int(sum(mf.mo_occ[active_space]))
num_elec_a   = (nelec_total + mol.spin) // 2
num_elec_b   = (nelec_total - mol.spin) // 2
nelec        = (num_elec_a, num_elec_b)

cas        = mcscf.CASCI(mf, norb, nelec)
mo         = cas.sort_mo(active_space, base=0)
h1e, nuc_energy = cas.get_h1cas(mo)
h2e        = ao2mo.restore(1, cas.get_h2cas(mo), norb)

print(f'Active space: norb={norb}  nelec={nelec}  nuc={nuc_energy:.8f}')

converged SCF energy = -108.835236570775
HF energy: -108.83523657
Active space: norb=16  nelec=(5, 5)  nuc=-76.23110254


## 4. CCSD Amplitudes for LUCJ Initialization

In [5]:
ccsd = cc.CCSD(mf, frozen=n_frozen).run()
t1, t2 = ccsd.t1, ccsd.t2
print(f'E(CCSD) = {ccsd.e_tot:.12f}  E_corr = {ccsd.e_corr}')
print(f't1 shape: {t1.shape}  t2 shape: {t2.shape}')

E(CCSD) = -109.0398256929948  E_corr = -0.2045891222202483
E(CCSD) = -109.039825692995  E_corr = -0.20458912222024833
t1 shape: (5, 11)  t2 shape: (5, 5, 11, 11)


## 5. Initialize QRMI Backend

Credentials are loaded from  (or already set by Slurm SPANK plugin).


In [6]:
load_dotenv(DEMO_DIR / '.env', override=False)

BACKEND_NAME = os.environ.get('QRMI_JOB_QPU_RESOURCES', qpu_config['backend_name'])
os.environ.setdefault('QRMI_JOB_QPU_RESOURCES', BACKEND_NAME)
os.environ.setdefault('QRMI_JOB_QPU_TYPES', 'ibm-quantum-compute-service')

backend = QRMIBackend({'backend_name': BACKEND_NAME})
backend.initialize()
backend.get_backend()

props = backend.get_backend_properties()
print(f'Backend : {props["backend_name"]}')
print(f'Type    : {props["resource_type"]}')

Backend : ibm_kingston
Type    : ResourceType.IBMQuantumComputeService


## 6. QPU Sampling (shared checkpoint)

The LUCJ circuit is submitted once.  Both SQD and TrimSQD post-processing
steps reuse the same  checkpoint — no extra QPU cost.

Set  to clear the checkpoint and submit a fresh job.


In [7]:
# Shared checkpoint directory — both solvers read from here
workflow_dir = DEMO_DIR / 'sqd_results' / 'N2_6-31G_trim'

# Use SQDSolver just for the QPU sampling step, then reuse counts for TrimSQD
sampler = SQDSolver(backend, config=sqd_config)

# This cell submits (or resumes) the QPU job and saves counts.npy
import numpy as np
from pathlib import Path
from quantum_fragment_methods.application.solvers.quantum_zoo.sqd import counts_to_bit_array

counts_file = Path(workflow_dir) / 'counts.npy'
job_id_file = Path(workflow_dir) / 'job_id.txt'

# Run QPU sampling (handles submit + poll + checkpoint)
result_std = sampler.solve(
    h1e=h1e,
    h2e=h2e,
    norb=norb,
    nelec=nelec,
    t1=t1,
    t2=t2,
    workflow_path=str(workflow_dir),
    wait_for_completion=True,
    force_resubmit=False,
)
print(f'Standard SQD — E = {result_std.energy + nuc_energy:.8f} Ha')

  QPU job submitted: daiqkgfi3e6s738pebs0
  Backend: ibm_kingston  checkpoint: /Users/thaddeuspellegrini/Code/QuantumAlgorithmEngineering/QDC_2026/quantum-fragment-methods/examples/notebook_demos/sqd_demos/sqd_results/N2_6-31G_trim/job_id.txt
  Polling job daiqkgfi3e6s738pebs0 every 30s (max 300s) ...
  00s] daiqkgfi3e6s738pebs0 — DONE
  ✓ Job completed (0m00s)
  SQD: 5 iter × 5 batches × 300 samples  backend=python
  SBD complete — E = -32.80140866 Ha
Standard SQD — E = -109.03251120 Ha


## 7. TrimSQD Post-Processing

Reload the same counts and run TrimSQD post-processing.
No QPU submission — reads directly from the existing .


In [8]:
# Load the saved counts
counts_raw = np.load(counts_file, allow_pickle=True).item()

# Re-run post-processing with TrimSQDSolver
trim_solver = TrimSQDSolver(backend, config=sqd_config)

result_trim = trim_solver._sbd_postprocessing(
    h1e=h1e,
    h2e=h2e,
    counts=counts_raw,
    norb=norb,
    nelec=nelec,
    workflow_path=Path(workflow_dir),
)
print(f'TrimSQD — E = {result_trim.energy + nuc_energy:.8f} Ha')

  TrimSQD: 5 iter × 5 batches × 300 samples  trim_ratio=0.5  TrimPolicy  backend=python
TrimSQD — E = -109.04251331 Ha


## 8. Results Comparison

In [9]:
CASCI_energy = -109.04667800  # from qiskit-addon-sqd tutorial

std_total  = result_std.energy  + nuc_energy
trim_total = result_trim.energy + nuc_energy

std_err  = abs(std_total  - CASCI_energy)
trim_err = abs(trim_total - CASCI_energy)

print(f'{"Method":<20}  {"Total energy (Ha)":>20}  {"vs CASCI (mHa)":>16}  {"vs CASCI (kcal/mol)":>22}')
print('-' * 82)
print(f'{"HF":<20}  {mf.e_tot + nuc_energy:>20.8f}  {(mf.e_tot + nuc_energy - CASCI_energy)*1000:>16.4f}  {(mf.e_tot + nuc_energy - CASCI_energy)*627.509:>22.4f}')
print(f'{"CASCI (reference)":<20}  {CASCI_energy:>20.8f}  {0.0:>16.4f}  {0.0:>22.4f}')
print(f'{"Standard SQD":<20}  {std_total:>20.8f}  {(std_total - CASCI_energy)*1000:>16.4f}  {(std_total - CASCI_energy)*627.509:>22.4f}')
print(f'{"TrimSQD":<20}  {trim_total:>20.8f}  {(trim_total - CASCI_energy)*1000:>16.4f}  {(trim_total - CASCI_energy)*627.509:>22.4f}')
print()
print(f'TrimSQD improvement over Standard SQD: {(std_err - trim_err)*1000:+.4f} mHa')

Method                   Total energy (Ha)    vs CASCI (mHa)     vs CASCI (kcal/mol)
----------------------------------------------------------------------------------
HF                           -185.06633911       -76019.6611             -47703.0215
CASCI (reference)            -109.04667800            0.0000                  0.0000
Standard SQD                 -109.03251120           14.1668                  8.8898
TrimSQD                      -109.04251331            4.1647                  2.6134

TrimSQD improvement over Standard SQD: +10.0021 mHa


## Notes

### When to use TrimSQD vs Standard SQD

| Scenario | Recommendation |
|----------|---------------|
| Small norb (≤ 10), many shots | Standard SQD — subspace is tractable |
| Large norb (> 14), limited shots | **TrimSQD** — screening reduces effective subspace |
| HPC + SBD solver | **TrimSQD** — SBD handles large merged subspaces efficiently |
| Laptop demo, norb > 16 | TrimSQD with  cap |

### TrimSQD parameters

- : fraction of strings retained from each batch (0.1–0.5 typical)
  - Lower → smaller merged subspace, faster but may miss important strings
  - Higher → larger merged subspace, more accurate but slower
- : hard cap regardless of 
  - Essential for large  on laptop to bound FCI dimension
- : more batches = broader screening pool = better coverage
  - TrimSQD benefits more from  than standard SQD

### SectorTrimPolicy

Switch  in the config to use , which
partitions the alpha and beta CI string arrays directly (making each spin
sector disjoint across batches) rather than partitioning the sampled
bitstrings.  This can improve coverage of the spin-resolved subspace.
